In [1]:
def load_data(file_name: str) :
    """"
        Helper functions that extracts the data from the input files.
        The format based on which it extracts can be seen at 'knapsack-20.txt'
        The first line holds the nr of total objects n, then n lines represeting 
        the index, value and weight of the values, then the max_weight is on the last line. 
    Args:
        file_name (str): Input file for extracting data

    Returns:
        int : The total number of objects 
        list[tuple[int, int]]: The results consists of the n tuples(represnting (value, weight) pairs)
        int : The maximum value for the total weight
    """
    weights_and_values = []
    with open(file_name) as f:
        total = int(f.readline())
        for _ in range(total) :
            values = f.readline().split()
            n2 = int(values[1])
            n3 = int(values[2])
            weights_and_values.append((n2, n3))
        
        max_value = int(f.readline())
    
    return total, weights_and_values, max_value

In [2]:
def compute_value_for_solution(solution :list, max_value_weight, weights_values:list[tuple]) :
    """For each solution generated it computes the total value accumulated and
    the weight it reaches , if the weight is valid it return the maximum one, otherwise -1.

    Args:
        solution (list): The list of size n representing the made out of 0s and 1s, representing
        whether the values are added or not.
        max_value_value (int): The maximum weight given as input
        weights_values (list[tuple]) : The input data for all the objects (value, weight) pairs
        
    Returns:
    int : Returns the total value if the weight is lower that the max_value_weight,
            otherwise returns -1
    """
    sum_values = 0
    sum_weights = 0
    
    for i in range(len(solution)) :
        sum_values+=solution[i]*weights_values[i][0]
        sum_weights+=solution[i]*weights_values[i][1]
        
    if sum_weights <= max_value_weight :
        return sum_values
    return -1

In [ ]:
import random
import math
import sys

def simulated_annealing_knapsack(filename: str, T_max: float = 10000,
                                  T_min: float = 0.1, alpha: float = 0.999,
                                  iterations_per_temp: int = 100, cooling='geometric'):
    """Generate a feasible solution then go for each temperature a number of iterations where
    we check the neighbout via bit flip if better value found we update the local maxima,
    otherwise we have a small probability where even a worse feasible solution gets chosen therefore
    giving chance to escaping the local maxima, only feasible solutions are taken into consideration.
    Via the parameters the algorithm is pretty customizable.

    Args:
        filename (str): _description_
        T_max (float, optional): _description_. Defaults to 10000.
        T_min (float, optional): _description_. Defaults to 0.1.
        alpha (float, optional): _description_. Defaults to 0.999.
        iterations_per_temp (int, optional): _description_. Defaults to 100.
        cooling (str, optional): _description_. Defaults to 'geometric'.

    Returns:
        _type_: _description_
    """
    

    total, weights_values, max_weight = load_data(filename)
    MAX_EXP = math.log(sys.float_info.max)

    c = [random.randint(0, 1) for _ in range(total)]
    c_fitness = compute_value_for_solution(c, max_weight, weights_values)
    
    while c_fitness == -1:
        c = [random.randint(0, 1) for _ in range(total)]
        c_fitness = compute_value_for_solution(c, max_weight, weights_values)

    best = c.copy()
    best_fitness = c_fitness

    T = T_max
    t = 0

    while T > T_min:
        for _ in range(iterations_per_temp):

            i = random.randint(0, total - 1)
            x = c.copy()
            x[i] = 1 - x[i]
            x_fitness = compute_value_for_solution(x, max_weight, weights_values)

            if x_fitness != -1:
                delta = x_fitness - c_fitness

                if delta > 0:
                    c = x
                    c_fitness = x_fitness

                else:
                    exponent = (delta) / T
                    if exponent > -MAX_EXP:
                        if random.random() < math.exp(exponent):
                            c = x
                            c_fitness = x_fitness

            if c_fitness > best_fitness:
                best = c.copy()
                best_fitness = c_fitness

        t += 1
        if cooling == 'geometric':
            T = alpha * T
        elif cooling == 'logarithmic':
            T = T / math.log(t + 1)
        elif cooling == 'linear':
            T = T / t

    return best_fitness

In [93]:
simulated_annealing_knapsack('../Lab1Assigment1/knapsack-20.txt', T_min=0.0000000001, cooling='logarithmic')

693

In [94]:
simulated_annealing_knapsack('../Lab1Assigment1/knapsack-20.txt', 
                             T_max=10000, T_min = 0.001, alpha=0.999, 
                             iterations_per_temp=40,cooling='geometric')

726

In [ ]:
def neighbours_visited(T_max, T_min, alpha, iterations_per_temp, cooling):
    """A function that computes for each cooling type and values for the hyperparams from
    the SA function how many neighbours are being visited

    Args:
        T_max (_type_): _description_
        T_min (_type_): _description_
        alpha (_type_): _description_
        iterations_per_temp (_type_): _description_
        cooling (_type_): _description_

    Returns:
        _type_: _description_
    """
    T = T_max
    t = 0
    while T > T_min:
        t += 1
        if cooling == 'geometric':
            T = alpha * T
        elif cooling == 'logarithmic':
            T = T / math.log(t + 1)
        elif cooling == 'linear':
            T = T / t
    return t * iterations_per_temp

In [ ]:
def generate_report(filename: str, output: str, 
                    iterations_values, T_max_values, T_min_values,
                    alpha_values, cooling):
    """We do a dedicated report function where we check how results vary in terms and relation
    to the iterations number, T_max, T_min, alpha and cooling type

    Args:
        filename (str): _description_
        output (str): _description_
        iterations_values (list): _description_
        sample_values (list): _description_
    """

    with open(output, 'a') as f:
        f.write(f"\nReport for {filename}\n")

    
    with open(output, 'a') as f:
        f.write(f"\nChecking for {cooling} cooling different values\n")
    for val in iterations_values:
        for t_max in T_max_values:
            for t_min in T_min_values :
                for alpha in alpha_values :
                    
                    temperature_steps = int(math.log(t_min / t_max) / math.log(alpha))
                    neighbours =  neighbours_visited(t_max, t_min, alpha, val, cooling)
                      
                    result = simulated_annealing_knapsack(filename, t_max, t_min, alpha, val, cooling)
                    with open(output, 'a') as f:
                        f.write(f"iterations={val}, T_max={t_max}, T_min={t_min}, alpha={alpha}, neighbours checked={neighbours} => result={result}\n")



In [103]:
generate_report('../Lab1Assigment1/knapsack-200.txt', 'report_knapsack_200.txt', 
                [50, 100, 200, 300],
                [100, 1000, 10000],
                [1, 0.1, 0.01],
                [0.9, 0.95, 0.99],
                'logarithmic')

In [100]:
generate_report('../Lab1Assigment1/knapsack-20.txt', 'report_knapsack_20.txt',
                iterations_values=[5, 10, 20, 40],
                T_max_values=[100, 1000, 10000, 100000],
                T_min_values=[0.1, 0.01, 0.001],
                alpha_values=[0.9, 0.95, 0.99, 0.999],
                cooling='geometric')